# DOE MASTER FULL PIPELINE — one-notebook ML + output figures

This is the single notebook to copy/run on the DOE Anaconda desktop. It combines the training pipeline and the trained-output review/export pipeline into one file.

Core inputs are the three normalized Excel workbooks for the four wells. Real measured NMR is not required; the notebook uses proxy features from the available well-log parameters. Optional moisture/temperature CSV context is searched locally, but the priority outputs are ML outputs produced after training/prediction.

**Run order:** setup → header scan → choose target if needed → main ML run → optional dataset-3 run → optional all-saturation run → ML output tables/figures.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, warnings

required = ["pandas", "numpy", "sklearn", "openpyxl", "joblib", "matplotlib"]
missing = []
for package in required:
    import_name = "sklearn" if package == "sklearn" else package
    try:
        __import__(import_name)
    except Exception:
        missing.append(package)
if missing:
    raise ImportError(f"Missing packages in Anaconda environment: {missing}")

import numpy as np
import pandas as pd
from joblib import dump, load
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

# -----------------------------
# Local DOE desktop paths
# -----------------------------
DATA_DIR = Path.home() / "Downloads" / "Northslopedatasets06052026"
WORKBOOKS = ("curated_dataset1.xlsx", "curated_dataset2.xlsx", "curated_dataset3.xlsx")
EXPECTED_WELL_COUNT = 4

# Project assumptions
NORMALIZED_INPUTS = True
USE_REAL_NMR_IF_PRESENT = False
MOISTURE_TEMPERATURE_CSV_NAME = None

# Model defaults
DEFAULT_MODEL_KIND = "baseline"   # baseline=random forest, mlp=small sklearn neural net
USE_SCALER = False                # normalized inputs; MLP forces scaling automatically
RANDOM_STATE = 42

# Local outputs. These stay on DOE desktop.
OUTPUT_ROOT = Path.cwd() / "outputs_runtime"
MODEL_ROOT = Path.cwd() / "models_runtime"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

print("Notebook folder:", Path.cwd())
print("Data folder:", DATA_DIR)
for workbook in WORKBOOKS:
    print(workbook, "FOUND" if (DATA_DIR / workbook).exists() else "MISSING")


## Chong-inspired ML result plot plan

The notebook creates **end-of-pipeline ML outputs**, not just input plots. The plot set is based on the type of ML evidence needed for a Chong-style hydrate saturation workflow:

1. **Prediction vs. reference saturation/class plot** — compares `y_pred` to `y_true` where the target is available.
2. **Residual histogram** — shows prediction error distribution where `y_true` exists.
3. **Metric comparison bars** — MAE/RMSE/R² or classification metrics by dataset/split.
4. **Feature importance plot** — for random-forest models, shows which features drive predictions.
5. **Predicted output vs. depth/well plot** — shows model output as a depth-profile style result.
6. **All-saturation target summary** — compares trained target runs from the all-saturation workflow.
7. **Prediction-file inventory** — documents which outputs exist without exposing full prediction rows by default.

These are meant for the research paper and slide deck. The full prediction CSVs and fitted models remain local under `outputs_runtime/` and `models_runtime/`.


In [ ]:
# -----------------------------
# Schema, aliases, and helper functions
# -----------------------------

REQUIRED_LOG_COLUMNS = ("well_alias", "depth_m", "gr_api", "rt_ohm_m", "rhob_g_cc")
OPTIONAL_LOG_COLUMNS = (
    "density_porosity_vv", "neutron_porosity_vv", "dt_us_ft", "dts_us_ft",
    "vp_km_s", "vs_km_s", "vp_m_s", "vs_m_s", "nmr_porosity_vv",
    "caliper_in", "temperature_c", "pressure_mpa",
)
CHONG_ML_FEATURE_COLUMNS = ("rhob_g_cc", "density_porosity_vv", "rt_ohm_m", "gr_api", "vp_km_s", "vs_km_s")

TARGET_ONLY_COLUMN_ALIASES = {
    "hydrate_saturation": ("Sgh", "S_h", "Sh", "Shyd", "NMR_SAT", "Hydrate Saturation", "hydrate_saturation_vv", "hydrate_sat"),
    "irreducible_or_residual_water_saturation": ("Swr", "S_wr", "Swirr", "irreducible_water_saturation_vv"),
    "phase_or_occurrence_label": ("interpreted phase label", "phase_label", "hydrate_phase", "hydrate_occurrence_label", "runtime_phase_label", "class", "label", "occurrence"),
}
CURVE_ALIASES = {
    "well_alias": ("well_alias", "WELL", "WELL_NAME", "Well Name", "UWI", "API", "Well"),
    "depth_m": ("depth_m", "DEPTH_M", "DEPTH", "MD", "MD_M", "TVD", "TVD_M", "DEPT"),
    "gr_api": ("gr_api", "GR", "GAMMA", "GAMMA_RAY", "Gamma Ray"),
    "rt_ohm_m": ("rt_ohm_m", "RT", "ILD", "RDEP", "RES", "RES_DEEP", "Deep formation resistivity"),
    "rhob_g_cc": ("rhob_g_cc", "RHOB", "DEN", "DENSITY", "Rho_b", "Density_gcpcc", "Density_gpcc", "bulk_density"),
    "density_porosity_vv": ("density_porosity_vv", "DPHI", "PHID", "DEN_POR", "Phi_porosity", "phi_den", "densityporosity"),
    "neutron_porosity_vv": ("neutron_porosity_vv", "NPHI", "TNPH", "NEUTRON_POR", "phi_neut"),
    "dt_us_ft": ("dt_us_ft", "DT", "DTC", "AC"),
    "dts_us_ft": ("dts_us_ft", "DTS", "DTSM"),
    "vp_km_s": ("vp_km_s", "VP_KM_S", "vpkms"),
    "vs_km_s": ("vs_km_s", "VS_KM_S", "vskms"),
    "vp_m_s": ("vp_m_s", "Vp", "VP", "VELP", "VP_M_S", "vpmps"),
    "vs_m_s": ("vs_m_s", "Vs", "VS", "VS1", "VELS", "VS_M_S", "vsmps"),
    "nmr_porosity_vv": ("nmr_porosity_vv", "NMRPHI", "TCMR", "CMRP", "phi_nmr", "nmrporosity"),
    "caliper_in": ("caliper_in", "CALI", "CALIPER", "caliper", "CAL1"),
    "temperature_c": ("temperature_c", "TEMP", "TEMPERATURE_C", "temperature"),
    "pressure_mpa": ("pressure_mpa", "PRESSURE", "PRESSURE_MPA", "PP_MPA"),
}
CLASSIFICATION_NAME_HINTS = ("occurrence", "class", "label", "phase", "hydratepresent")
REGRESSION_NAME_HINTS = ("saturation", "sat", "sgh", "shyd", "nmr_sat")
IDENTIFIER_AND_CONTEXT_COLUMNS = {
    "well_alias", "depth_m", "source_dataset", "dataset_file",
    "source_sheet", "split", "row_index", "source_workbook", "source_sheet_name",
}
CONTEXT_OR_HELPER_EXACT = {"index", "row", "rowindex", "depth", "depthm", "depthft", "dept", "md", "tvd"}
CONTEXT_OR_HELPER_PARTS = ("depth", "unit", "units")
MAX_MODEL_ABS_VALUE = 1.0e12

def normalize_header_name(value):
    return "".join(character for character in str(value).lower() if character.isalnum())

def sanitize_label(value, max_len=100):
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value).strip()).strip("._-")
    return cleaned[:max_len] or "run"

def clean_numeric_series(values):
    numeric = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return numeric.mask(numeric.abs() > MAX_MODEL_ABS_VALUE)

def clean_numeric_frame(frame):
    cleaned = frame.copy()
    for column in cleaned.columns:
        series = cleaned[column].astype(int) if pd.api.types.is_bool_dtype(cleaned[column]) else cleaned[column]
        cleaned[column] = clean_numeric_series(series)
    return cleaned

def target_alias_lookup():
    lookup = {}
    for family, aliases in TARGET_ONLY_COLUMN_ALIASES.items():
        for alias in aliases:
            lookup[normalize_header_name(alias)] = (family, alias)
    return lookup

def canonical_name_for_header(header):
    normalized = normalize_header_name(header)
    for canonical, aliases in CURVE_ALIASES.items():
        if normalized == normalize_header_name(canonical):
            return canonical
        for alias in aliases:
            if normalized == normalize_header_name(alias):
                return canonical
    return None

def target_like_column(column):
    normalized = normalize_header_name(column)
    if normalized in target_alias_lookup():
        return True
    if normalized in {"class", "label", "target", "y", "occurrence", "phase", "hydratepresent"}:
        return True
    has_hydrate = "hydrate" in normalized or "sgh" in normalized or normalized.startswith("sh")
    has_target_word = any(part in normalized for part in ("sat", "saturation", "occur", "class", "label", "phase"))
    return has_hydrate and has_target_word

def saturation_like_column(column):
    normalized = normalize_header_name(column)
    if any(word in normalized for word in ("porosity", "density", "sample", "station", "status")):
        return False
    exact = {"sgh", "sh", "shyd", "hydratesat", "hydratesaturation", "nmrsat", "swr", "sw", "swi", "swirr"}
    return normalized in exact or "sat" in normalized or "saturation" in normalized

def is_context_or_helper_column(column):
    normalized = normalize_header_name(column)
    canonical = canonical_name_for_header(column)
    return (
        normalized.startswith("unnamed")
        or normalized in CONTEXT_OR_HELPER_EXACT
        or any(part in normalized for part in CONTEXT_OR_HELPER_PARTS)
        or canonical in {"well_alias", "depth_m"}
    )


In [ ]:
# -----------------------------
# Loading, validation, feature engineering
# -----------------------------

def standardize_curve_columns(frame):
    lookup = {normalize_header_name(column): column for column in frame.columns}
    rename_map = {}
    for canonical, aliases in CURVE_ALIASES.items():
        for alias in aliases:
            actual = lookup.get(normalize_header_name(alias))
            if actual is not None:
                rename_map[actual] = canonical
                break
    standardized = frame.rename(columns=rename_map).copy()
    for column in standardized.columns:
        if column == "well_alias":
            continue
        numeric = clean_numeric_series(standardized[column])
        if numeric.notna().any():
            standardized[column] = numeric
    if "well_alias" in standardized:
        standardized["well_alias"] = standardized["well_alias"].astype(str)
    return standardized

def read_first_nonempty_excel_sheet(path, sheet_name=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    with pd.ExcelFile(path) as excel:
        sheet_names = list(excel.sheet_names)
    candidates = [sheet_name] if sheet_name else sheet_names
    for candidate in candidates:
        if candidate not in sheet_names:
            raise ValueError(f"Sheet {candidate!r} missing from {path.name}")
        frame = pd.read_excel(path, sheet_name=candidate)
        if not frame.empty and len(frame.columns):
            return frame, str(candidate)
    return pd.read_excel(path, sheet_name=sheet_names[0]), str(sheet_names[0])

def load_workbook_dataset(data_dir, file_name, split, sheet_name=None):
    raw, used_sheet = read_first_nonempty_excel_sheet(Path(data_dir) / file_name, sheet_name)
    frame = standardize_curve_columns(raw)
    frame["source_dataset"] = Path(file_name).stem
    frame["dataset_file"] = file_name
    frame["source_sheet"] = used_sheet
    frame["split"] = split
    frame["row_index"] = np.arange(len(frame))
    if "well_alias" not in frame:
        frame["well_alias"] = Path(file_name).stem
    return {"label": Path(file_name).stem, "split": split, "path": str(Path(data_dir) / file_name), "sheet_name": used_sheet, "frame": frame}

def load_three_dataset_workbooks(data_dir=DATA_DIR, train_file="curated_dataset1.xlsx", test_files=("curated_dataset2.xlsx", "curated_dataset3.xlsx"), sheet_name=None):
    datasets = [load_workbook_dataset(data_dir, train_file, "train", sheet_name)]
    for file_name in test_files:
        if (Path(data_dir) / file_name).exists():
            datasets.append(load_workbook_dataset(data_dir, file_name, "test", sheet_name))
    return datasets

def validate_log_table(logs):
    issues = []
    for column in REQUIRED_LOG_COLUMNS:
        if column not in logs.columns:
            issues.append({"Severity": "error", "Column": column, "Message": "Missing required log column."})
    for column in logs.columns:
        family_match = target_alias_lookup().get(normalize_header_name(column))
        if family_match:
            issues.append({"Severity": "error", "Column": str(column), "Message": f"Target-only/calibration column detected in feature-side input: {family_match[0]} matched {family_match[1]}."})
    if logs.empty:
        issues.append({"Severity": "error", "Column": "table", "Message": "No rows loaded."})
    if "well_alias" in logs and logs["well_alias"].isna().any():
        issues.append({"Severity": "error", "Column": "well_alias", "Message": "Missing well aliases."})
    if "well_alias" in logs:
        well_count = int(logs["well_alias"].nunique())
        if well_count != EXPECTED_WELL_COUNT:
            issues.append({"Severity": "warning", "Column": "well_alias", "Message": f"Expected {EXPECTED_WELL_COUNT} wells; detected {well_count}."})
    if {"well_alias", "depth_m"}.issubset(logs.columns):
        for alias, well in logs.groupby("well_alias", dropna=False):
            depth = clean_numeric_series(well["depth_m"])
            if not depth.is_monotonic_increasing:
                issues.append({"Severity": "warning", "Column": "depth_m", "Message": f"Depth is not monotonic for well {alias}."})
            if depth.duplicated().any():
                issues.append({"Severity": "warning", "Column": "depth_m", "Message": f"Duplicate depths found for well {alias}."})
    status = "blocked" if any(issue["Severity"] == "error" for issue in issues) else "ready"
    return status, pd.DataFrame(issues)

def add_standard_features(logs):
    features = logs.copy()
    if "gr_api" in features:
        features["vshale_proxy"] = clean_numeric_series(features["gr_api"]).clip(0, 1) if NORMALIZED_INPUTS else ((clean_numeric_series(features["gr_api"]) - 30) / (105 - 30)).clip(0, 1)
    if "rhob_g_cc" in features and "density_porosity_vv" not in features:
        rho = clean_numeric_series(features["rhob_g_cc"])
        features["density_porosity_vv"] = (1 - rho).clip(0, 1) if NORMALIZED_INPUTS else ((2.65 - rho) / (2.65 - 1.03)).clip(0, 0.7)
    if not NORMALIZED_INPUTS:
        if "dt_us_ft" in features and "vp_km_s" not in features:
            features["vp_km_s"] = 304.8 / clean_numeric_series(features["dt_us_ft"])
        if "dts_us_ft" in features and "vs_km_s" not in features:
            features["vs_km_s"] = 304.8 / clean_numeric_series(features["dts_us_ft"])
    if "vp_m_s" in features and "vp_km_s" not in features:
        features["vp_km_s"] = clean_numeric_series(features["vp_m_s"]) / (1000.0 if not NORMALIZED_INPUTS else 1.0)
    if "vs_m_s" in features and "vs_km_s" not in features:
        features["vs_km_s"] = clean_numeric_series(features["vs_m_s"]) / (1000.0 if not NORMALIZED_INPUTS else 1.0)
    if {"vp_km_s", "vs_km_s"}.issubset(features.columns):
        features["vp_vs_ratio"] = clean_numeric_series(features["vp_km_s"]) / clean_numeric_series(features["vs_km_s"]).replace(0, np.nan)
    if {"rhob_g_cc", "vs_km_s"}.issubset(features.columns):
        features["shear_modulus_proxy"] = clean_numeric_series(features["rhob_g_cc"]) * clean_numeric_series(features["vs_km_s"]) ** 2
    if {"rhob_g_cc", "vp_km_s", "vs_km_s"}.issubset(features.columns):
        features["bulk_modulus_proxy"] = clean_numeric_series(features["rhob_g_cc"]) * (clean_numeric_series(features["vp_km_s"]) ** 2 - (4 / 3) * clean_numeric_series(features["vs_km_s"]) ** 2)
    # Project-specific NMR handling: no real NMR column required by default.
    if USE_REAL_NMR_IF_PRESENT and {"density_porosity_vv", "nmr_porosity_vv"}.issubset(features.columns):
        features["nmr_density_separation_vv"] = clean_numeric_series(features["density_porosity_vv"]) - clean_numeric_series(features["nmr_porosity_vv"])
        features["nmr_density_hydrate_proxy"] = (features["nmr_density_separation_vv"] / clean_numeric_series(features["density_porosity_vv"]).clip(0.01)).clip(0, 1)
    else:
        if {"density_porosity_vv", "rt_ohm_m"}.issubset(features.columns):
            phi = clean_numeric_series(features["density_porosity_vv"]).clip(0.01)
            rt = clean_numeric_series(features["rt_ohm_m"]).clip(0.01)
            features["log_proxy_hydrate_index"] = (phi * rt).clip(0, 1) if NORMALIZED_INPUTS else (1 - ((0.12 / (phi.clip(0.04) ** 2 * rt)) ** 0.5)).clip(0, 1)
        if {"density_porosity_vv", "vp_km_s"}.issubset(features.columns):
            features["density_sonic_proxy_index"] = (clean_numeric_series(features["density_porosity_vv"]) * clean_numeric_series(features["vp_km_s"])).clip(0, 1)
    if "caliper_in" in features:
        caliper = clean_numeric_series(features["caliper_in"])
        threshold = caliper.groupby(features["well_alias"]).transform(lambda values: values.quantile(0.95)) if "well_alias" in features else pd.Series(caliper.quantile(0.95), index=features.index)
        features["caliper_washout_flag"] = caliper.ge(threshold) & caliper.notna()
    available = [column for column in CHONG_ML_FEATURE_COLUMNS if column in features.columns]
    features["chong_ml_available_feature_count"] = features[available].notna().sum(axis=1) if available else 0
    features["chong_ml_complete_case_flag"] = features[available].notna().all(axis=1) if available else False
    if "caliper_washout_flag" in features:
        features["chong_ml_complete_case_flag"] &= ~features["caliper_washout_flag"]
    return features.replace([np.inf, -np.inf], np.nan)

def make_feature_matrix(frame, target_columns):
    features = add_standard_features(frame)
    selected, inventory, audit = [], [], []
    for column in features.columns:
        name = str(column)
        canonical = canonical_name_for_header(name)
        duplicate = canonical is not None and canonical != name and canonical in features.columns
        reason = ""
        if name in target_columns or target_like_column(name):
            reason = "target_or_target_like_leakage"
        elif name in IDENTIFIER_AND_CONTEXT_COLUMNS:
            reason = "identifier_or_runtime_context"
        elif is_context_or_helper_column(name):
            reason = "context_depth_unit_or_spreadsheet_helper"
        elif duplicate:
            reason = f"raw_alias_duplicate_of_{canonical}"
        elif name == "nmr_porosity_vv" and not USE_REAL_NMR_IF_PRESENT:
            reason = "real_nmr_disabled_for_current_project_state"
        elif not pd.api.types.is_numeric_dtype(features[column]) and not pd.api.types.is_bool_dtype(features[column]):
            reason = "non_numeric_or_categorical"
        values = None
        if not reason:
            source = features[column].astype(int) if pd.api.types.is_bool_dtype(features[column]) else features[column]
            values = clean_numeric_series(source)
            if values.notna().sum() == 0:
                reason = "numeric_empty_after_cleaning"
        audit.append({"column": name, "decision": "excluded" if reason else "included", "reason": reason, "canonical_header": canonical or "", "numeric_rows": int(values.notna().sum()) if values is not None else 0})
        if reason:
            continue
        features[name] = values
        selected.append(name)
        inventory.append({"feature_column": name, "non_null_rows": int(values.notna().sum()), "coverage_fraction": round(float(values.notna().mean()), 4), "minimum": float(values.min()), "maximum": float(values.max())})
    X = clean_numeric_frame(features[selected].copy()) if selected else pd.DataFrame(index=features.index)
    return X, pd.DataFrame(inventory), pd.DataFrame(audit)


In [ ]:
# -----------------------------
# Header scan, target detection, and model pipeline
# -----------------------------

def scan_three_dataset_headers(data_dir=DATA_DIR, files=WORKBOOKS, sample_rows=25, output_root=OUTPUT_ROOT, run_label=None):
    label = sanitize_label(run_label or "header_scan_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    out = Path(output_root) / label
    out.mkdir(parents=True, exist_ok=True)
    sheet_rows, column_rows, missing = [], [], []
    for file_name in files:
        path = Path(data_dir) / file_name
        if not path.exists():
            missing.append(str(path))
            continue
        with pd.ExcelFile(path) as excel:
            sheet_names = list(excel.sheet_names)
        for sheet_name in sheet_names:
            sample = pd.read_excel(path, sheet_name=sheet_name, nrows=sample_rows)
            header = pd.read_excel(path, sheet_name=sheet_name, nrows=0)
            sheet_rows.append({"workbook": file_name, "sheet_name": sheet_name, "sampled_rows": len(sample), "column_count": len(header.columns), "has_target_like_header": any(target_like_column(c) for c in header.columns)})
            for position, column in enumerate(header.columns, start=1):
                values = sample[column] if column in sample else pd.Series(dtype=object)
                numeric = clean_numeric_series(values)
                role = "possible_target_review" if target_like_column(column) else ("identifier_or_depth_axis" if canonical_name_for_header(column) in {"well_alias", "depth_m"} else ("candidate_feature_or_context" if canonical_name_for_header(column) else "unmapped_review"))
                task = "classification" if role == "possible_target_review" and any(h in normalize_header_name(column) for h in CLASSIFICATION_NAME_HINTS) else ("regression" if saturation_like_column(column) else "")
                column_rows.append({"workbook": file_name, "sheet_name": sheet_name, "column_position": position, "original_header": str(column), "normalized_header": normalize_header_name(column), "canonical_header": canonical_name_for_header(column) or "", "role_hint": role, "non_null_sample_rows": int(values.notna().sum()), "numeric_sample_rows": int(numeric.notna().sum()), "unique_sample_values": int(values.dropna().nunique()), "suggested_target_task": task})
    sheets = pd.DataFrame(sheet_rows)
    columns = pd.DataFrame(column_rows)
    target_hints = columns[columns["role_hint"].str.contains("target", case=False, na=False)].copy() if not columns.empty else pd.DataFrame()
    sheets.to_csv(out / "workbook_sheet_inventory.csv", index=False)
    columns.to_csv(out / "workbook_column_inventory.csv", index=False)
    target_hints.to_csv(out / "target_header_hints.csv", index=False)
    manifest = {"generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "missing_files": missing, "run_dir": str(out), "target_hint_count": int(len(target_hints))}
    (out / "header_scan_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest

def detect_target_candidates(frame):
    rows = []
    lookup = target_alias_lookup()
    for column in frame.columns:
        normalized = normalize_header_name(column)
        family = lookup.get(normalized, ("", ""))[0]
        if family or target_like_column(column):
            values = frame[column]
            numeric = clean_numeric_series(values)
            task = "classification" if family == "phase_or_occurrence_label" or any(h in normalized for h in CLASSIFICATION_NAME_HINTS) else ("regression" if numeric.notna().sum() > 0 else "classification")
            rows.append({"column": str(column), "normalized_column": normalized, "target_family": family or "target_like_unregistered", "task_hint": task, "non_null_rows": int(values.notna().sum()), "numeric_non_null_rows": int(numeric.notna().sum()), "unique_non_null_values": int(values.dropna().nunique())})
    return pd.DataFrame(rows)

def find_column_by_request(frame, requested):
    if requested in frame.columns:
        return requested
    normalized = normalize_header_name(requested)
    for column in frame.columns:
        if normalize_header_name(column) == normalized:
            return str(column)
    return None

def infer_task_for_column(frame, column, requested_task="auto"):
    if requested_task != "auto":
        return requested_task
    normalized = normalize_header_name(column)
    if any(hint in normalized for hint in CLASSIFICATION_NAME_HINTS):
        return "classification"
    if any(hint in normalized for hint in REGRESSION_NAME_HINTS):
        return "regression"
    numeric = clean_numeric_series(frame[column])
    return "regression" if numeric.notna().sum() >= max(3, int(frame[column].notna().sum() * 0.8)) else "classification"

def choose_target(train_frame, requested_target="auto", requested_task="auto"):
    candidates = detect_target_candidates(train_frame)
    selected = None
    reason = ""
    if requested_target != "auto":
        selected = find_column_by_request(train_frame, requested_target)
        if selected is None:
            raise ValueError(f"Requested target {requested_target!r} was not found in the training workbook.")
        reason = "target requested by notebook variable"
    elif not candidates.empty:
        order = candidates.copy()
        order["priority"] = order["target_family"].map({"hydrate_saturation": 0, "phase_or_occurrence_label": 1, "target_like_unregistered": 2, "irreducible_or_residual_water_saturation": 3}).fillna(9)
        order = order.sort_values(["priority", "non_null_rows", "unique_non_null_values"], ascending=[True, False, False])
        selected = str(order.iloc[0]["column"])
        reason = "first usable target-like column by project priority"
    if selected is None:
        return {"column": None, "task": "readiness_only", "family": "none_detected", "reason": "no target-like column detected"}, candidates
    task = infer_task_for_column(train_frame, selected, requested_task)
    family = target_alias_lookup().get(normalize_header_name(selected), ("target_like_unregistered", ""))[0]
    if not candidates.empty:
        candidates["selected"] = candidates["column"].astype(str).eq(selected)
    return {"column": selected, "task": task, "family": family, "reason": reason}, candidates

def build_model_pipeline(task, model_kind=DEFAULT_MODEL_KIND):
    if model_kind == "mlp":
        model = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=600, random_state=RANDOM_STATE) if task == "classification" else MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=600, random_state=RANDOM_STATE)
    else:
        model = RandomForestClassifier(n_estimators=250, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE) if task == "classification" else RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=RANDOM_STATE)
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if USE_SCALER or model_kind == "mlp":
        steps.append(("minmax_scaler", MinMaxScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

def evaluate_predictions(y_true, y_pred, task):
    if task == "classification":
        return {"rows_scored": int(len(y_true)), "accuracy": float(accuracy_score(y_true, y_pred)), "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)), "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0))}
    y_num = clean_numeric_series(y_true)
    p_num = clean_numeric_series(pd.Series(y_pred))
    return {"rows_scored": int(len(y_num)), "mae": float(mean_absolute_error(y_num, p_num)), "rmse": float(np.sqrt(mean_squared_error(y_num, p_num))), "r2": float(r2_score(y_num, p_num)) if len(y_num) >= 2 else None}

def prepare_supervised_table(frame, target, feature_columns, target_columns):
    X_all, _, _ = make_feature_matrix(frame, target_columns)
    for column in feature_columns:
        if column not in X_all:
            X_all[column] = np.nan
    X = clean_numeric_frame(X_all[feature_columns].copy())
    y = clean_numeric_series(frame[target["column"]]) if target["task"] == "regression" else frame[target["column"]].astype("string")
    mask = y.notna() & X.notna().any(axis=1)
    return X.loc[mask].reset_index(drop=True), y.loc[mask].reset_index(drop=True), frame.loc[mask].reset_index(drop=True)

def prepare_feature_only_table(frame, feature_columns, target_columns):
    X_all, _, _ = make_feature_matrix(frame, target_columns)
    for column in feature_columns:
        if column not in X_all:
            X_all[column] = np.nan
    X = clean_numeric_frame(X_all[feature_columns].copy())
    mask = X.notna().any(axis=1)
    return X.loc[mask].reset_index(drop=True), frame.loc[mask].reset_index(drop=True)

def prediction_frame(source_frame, y_true, y_pred, target_column, task, model=None, X=None):
    output = pd.DataFrame({
        "source_dataset": source_frame.get("source_dataset", pd.Series([""] * len(source_frame))).astype(str),
        "dataset_file": source_frame.get("dataset_file", pd.Series([""] * len(source_frame))).astype(str),
        "well_alias": source_frame.get("well_alias", pd.Series([""] * len(source_frame))).astype(str),
        "row_index": source_frame.get("row_index", pd.Series(range(len(source_frame)))),
        "target_column": target_column,
        "y_pred": y_pred,
    })
    if "depth_m" in source_frame:
        output["depth_m"] = source_frame["depth_m"]
    if y_true is not None:
        output["y_true"] = y_true.to_numpy()
    if task == "classification" and model is not None and X is not None and hasattr(model, "predict_proba"):
        try:
            output["y_pred_probability_max"] = model.predict_proba(X).max(axis=1)
        except Exception:
            pass
    return output

def dataset_inventory_frame(datasets):
    rows = []
    for dataset in datasets:
        frame = dataset["frame"]
        depth = clean_numeric_series(frame["depth_m"]) if "depth_m" in frame else pd.Series(dtype=float)
        rows.append({"source_dataset": dataset["label"], "split": dataset["split"], "file_name": Path(dataset["path"]).name, "sheet_name": dataset["sheet_name"], "rows": len(frame), "columns": len(frame.columns), "wells": int(frame["well_alias"].nunique()) if "well_alias" in frame else 0, "depth_min": float(depth.min()) if depth.notna().any() else None, "depth_max": float(depth.max()) if depth.notna().any() else None})
    return pd.DataFrame(rows)

def run_three_dataset_pipeline(data_dir=DATA_DIR, train_file="curated_dataset1.xlsx", test_files=("curated_dataset2.xlsx", "curated_dataset3.xlsx"), requested_target="auto", requested_task="auto", model_kind=DEFAULT_MODEL_KIND, run_label=None):
    label = sanitize_label(run_label or "three_dataset_ml_run_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    run_dir = OUTPUT_ROOT / label
    model_dir = MODEL_ROOT / label
    run_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    datasets = load_three_dataset_workbooks(data_dir, train_file, test_files)
    train_dataset = datasets[0]
    target, target_candidates = choose_target(train_dataset["frame"], requested_target, requested_task)
    target_columns = set(target_candidates.get("column", pd.Series(dtype=str)).astype(str)) if not target_candidates.empty else set()
    if target["column"]:
        target_columns.add(target["column"])
    dataset_inventory_frame(datasets).to_csv(run_dir / "dataset_inventory.csv", index=False)
    target_candidates.to_csv(run_dir / "target_detection.csv", index=False)
    feature_matrix, feature_inventory, feature_audit = make_feature_matrix(train_dataset["frame"], target_columns)
    feature_inventory.to_csv(run_dir / "feature_columns.csv", index=False)
    feature_audit.to_csv(run_dir / "feature_policy_audit.csv", index=False)
    readiness_rows = []
    for dataset in datasets:
        status, issues = validate_log_table(dataset["frame"].drop(columns=list(target_columns), errors="ignore"))
        if issues.empty:
            issues = pd.DataFrame([{"Severity": "info", "Column": "table", "Message": "Feature-side log table passed current checks."}])
        issues.insert(0, "source_dataset", dataset["label"])
        issues.insert(1, "split", dataset["split"])
        issues.insert(2, "status", status)
        readiness_rows.append(issues)
    pd.concat(readiness_rows, ignore_index=True).to_csv(run_dir / "schema_readiness.csv", index=False)
    feature_columns = feature_matrix.columns.astype(str).tolist()
    train_metrics, test_metrics, written = [], [], {}
    status, blocked_reason = "readiness_only", ""
    if target["column"] is None:
        blocked_reason = target["reason"]
    elif not feature_columns:
        blocked_reason = "no numeric non-target feature columns detected after leakage/context exclusions"
    else:
        X_train, y_train, train_rows = prepare_supervised_table(train_dataset["frame"], target, feature_columns, target_columns)
        if len(X_train) < 3:
            blocked_reason = "fewer than three target-bearing training rows after preprocessing"
        elif target["task"] == "classification" and y_train.nunique() < 2:
            blocked_reason = "classification target has fewer than two classes"
        else:
            model = build_model_pipeline(target["task"], model_kind)
            model.fit(X_train, y_train)
            train_pred = model.predict(X_train)
            metrics = evaluate_predictions(y_train, train_pred, target["task"])
            metrics.update({"dataset": train_dataset["label"], "split": "train", "task": target["task"], "target_column": target["column"], "model_kind": model_kind, "feature_count": len(feature_columns)})
            train_metrics.append(metrics)
            train_path = run_dir / f"predictions_{sanitize_label(train_dataset['label'])}.csv"
            prediction_frame(train_rows, y_train, train_pred, target["column"], target["task"], model, X_train).to_csv(train_path, index=False)
            written["train_predictions"] = str(train_path)
            for dataset in datasets[1:]:
                if target["column"] in dataset["frame"].columns:
                    X_test, y_test, test_rows = prepare_supervised_table(dataset["frame"], target, feature_columns, target_columns)
                else:
                    X_test, test_rows = prepare_feature_only_table(dataset["frame"], feature_columns, target_columns)
                    y_test = None
                if X_test.empty:
                    test_metrics.append({"dataset": dataset["label"], "split": "test", "status": "blocked", "blocked_reason": "no feature rows available", "rows_scored": 0})
                    continue
                pred = model.predict(X_test)
                prediction_path = run_dir / f"predictions_{sanitize_label(dataset['label'])}.csv"
                prediction_frame(test_rows, y_test, pred, target["column"], target["task"], model, X_test).to_csv(prediction_path, index=False)
                written[f"predictions_{dataset['label']}"] = str(prediction_path)
                if y_test is not None and len(y_test):
                    m = evaluate_predictions(y_test, pred, target["task"])
                    m.update({"dataset": dataset["label"], "split": "test", "status": "scored", "task": target["task"], "target_column": target["column"], "feature_count": len(feature_columns)})
                    test_metrics.append(m)
                else:
                    test_metrics.append({"dataset": dataset["label"], "split": "test", "status": "predicted_unlabeled", "rows_scored": int(len(X_test)), "target_column": target["column"], "feature_count": len(feature_columns)})
            model_path = model_dir / "model.joblib"
            dump({"model": model, "feature_columns": feature_columns, "target": target, "model_kind": model_kind, "normalized_inputs": NORMALIZED_INPUTS, "real_nmr_used": USE_REAL_NMR_IF_PRESENT}, model_path)
            written["model"] = str(model_path)
            status = "trained"
    pd.DataFrame(train_metrics).to_csv(run_dir / "train_metrics.csv", index=False)
    pd.DataFrame(test_metrics).to_csv(run_dir / "test_metrics.csv", index=False)
    written.update({"dataset_inventory": str(run_dir / "dataset_inventory.csv"), "schema_readiness": str(run_dir / "schema_readiness.csv"), "target_detection": str(run_dir / "target_detection.csv"), "feature_columns": str(run_dir / "feature_columns.csv"), "feature_policy_audit": str(run_dir / "feature_policy_audit.csv"), "train_metrics": str(run_dir / "train_metrics.csv"), "test_metrics": str(run_dir / "test_metrics.csv")})
    manifest = {"generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "status": status, "blocked_reason": blocked_reason, "train_file": train_file, "test_files": list(test_files), "target": target, "model_kind": model_kind, "feature_count": len(feature_columns), "feature_columns": feature_columns, "outputs": written, "assumptions": {"expected_well_count": EXPECTED_WELL_COUNT, "normalized_inputs": NORMALIZED_INPUTS, "real_nmr_required": False}, "guardrail": "approved rows, predictions, and model files stay local unless reduced to public-safe summaries"}
    (run_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    return {"status": status, "blocked_reason": blocked_reason, "run_dir": str(run_dir), "model_dir": str(model_dir), "target": target, "feature_count": len(feature_columns), "outputs": written}


In [ ]:
# -----------------------------
# All-saturation target workflow
# -----------------------------

def read_workbook_sheets(data_dir=DATA_DIR, workbook_names=WORKBOOKS):
    sheets = []
    for workbook in workbook_names:
        path = Path(data_dir) / workbook
        if not path.exists():
            continue
        with pd.ExcelFile(path) as excel:
            sheet_names = list(excel.sheet_names)
        for sheet_name in sheet_names:
            frame = standardize_curve_columns(pd.read_excel(path, sheet_name=sheet_name))
            frame["source_workbook"] = workbook
            frame["source_sheet_name"] = str(sheet_name)
            frame["row_index"] = np.arange(len(frame))
            if "well_alias" not in frame:
                frame["well_alias"] = Path(workbook).stem
            sheets.append({"workbook": workbook, "sheet": str(sheet_name), "frame": frame})
    return sheets

def saturation_target_inventory(sheets):
    rows = []
    for sheet in sheets:
        frame = sheet["frame"]
        for column in frame.columns:
            if saturation_like_column(column):
                values = clean_numeric_series(frame[column])
                rows.append({"workbook": sheet["workbook"], "sheet_name": sheet["sheet"], "target_column": str(column), "numeric_rows": int(values.notna().sum()), "unique_values": int(values.dropna().nunique()), "minimum": float(values.min()) if values.notna().any() else None, "maximum": float(values.max()) if values.notna().any() else None})
    return pd.DataFrame(rows)

def find_feature_source_for_target(target_sheet, target_column, sheets):
    target_frame = target_sheet["frame"]
    saturation_columns = {str(column) for column in target_frame.columns if saturation_like_column(column)}
    same_features, _, _ = make_feature_matrix(target_frame, saturation_columns)
    if not same_features.empty:
        return target_frame.copy(), target_sheet["workbook"], "same_sheet"
    scored_candidates = []
    for candidate in sheets:
        candidate_frame = candidate["frame"]
        candidate_saturation = {str(column) for column in candidate_frame.columns if saturation_like_column(column)}
        candidate_features, _, _ = make_feature_matrix(candidate_frame, candidate_saturation)
        if candidate_features.empty:
            continue
        score = len(candidate_features.columns) + (100 if len(candidate_frame) == len(target_frame) else 0) + (50 if candidate["sheet"] == target_sheet["sheet"] else 0)
        scored_candidates.append((score, candidate))
    if not scored_candidates:
        return None, "", "blocked_no_feature_source"
    best = sorted(scored_candidates, key=lambda item: item[0], reverse=True)[0][1]
    if len(best["frame"]) != len(target_frame):
        return None, best["workbook"], "blocked_row_count_mismatch"
    aligned = best["frame"].reset_index(drop=True).copy()
    aligned[target_column] = target_frame[target_column].reset_index(drop=True)
    return aligned, best["workbook"], "row_order_aligned"

def fit_and_predict_all_saturations(data_dir=DATA_DIR, workbook_names=WORKBOOKS, output_dir=None, min_training_rows=5):
    output_dir = Path(output_dir) if output_dir else OUTPUT_ROOT / ("multi_saturation_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    output_dir.mkdir(parents=True, exist_ok=True)
    sheets = read_workbook_sheets(data_dir, workbook_names)
    target_inventory = saturation_target_inventory(sheets)
    target_inventory.to_csv(output_dir / "saturation_target_inventory.csv", index=False)
    pd.DataFrame([{"workbook": s["workbook"], "sheet_name": s["sheet"], "rows": len(s["frame"]), "columns": len(s["frame"].columns)} for s in sheets]).to_csv(output_dir / "sheet_inventory.csv", index=False)
    if target_inventory.empty:
        (output_dir / "run_summary.csv").write_text("status,message\nblocked,no saturation targets found\n", encoding="utf-8")
        return {"status": "blocked", "message": "no saturation targets found", "output_dir": str(output_dir)}
    summary_rows, feature_rows, audit_frames = [], [], []
    for _, target in target_inventory.iterrows():
        target_sheet = next(sheet for sheet in sheets if sheet["workbook"] == target["workbook"] and sheet["sheet"] == target["sheet_name"])
        target_column = str(target["target_column"])
        target_id = sanitize_label(f"{target['workbook']}_{target['sheet_name']}_{target_column}")
        training_frame, feature_source, alignment_method = find_feature_source_for_target(target_sheet, target_column, sheets)
        if training_frame is None:
            summary_rows.append({"target_id": target_id, "target_column": target_column, "target_workbook": target["workbook"], "target_sheet": target["sheet_name"], "status": "blocked", "alignment_method": alignment_method, "feature_source": feature_source, "training_rows": 0})
            continue
        saturation_columns = {str(column) for column in training_frame.columns if saturation_like_column(column)}
        X_all, feature_inventory, audit = make_feature_matrix(training_frame, saturation_columns)
        audit["target_id"] = target_id
        audit_frames.append(audit)
        y = clean_numeric_series(training_frame[target_column])
        mask = y.notna() & X_all.notna().any(axis=1)
        X_train = X_all.loc[mask].reset_index(drop=True)
        y_train = y.loc[mask].reset_index(drop=True)
        if len(X_train) < min_training_rows:
            summary_rows.append({"target_id": target_id, "target_column": target_column, "target_workbook": target["workbook"], "target_sheet": target["sheet_name"], "status": "blocked", "alignment_method": alignment_method, "feature_source": feature_source, "training_rows": int(len(X_train)), "blocked_reason": "not enough target-bearing training rows"})
            continue
        feature_columns = list(X_train.columns)
        model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=RANDOM_STATE))])
        model.fit(X_train[feature_columns], y_train)
        train_pred = model.predict(X_train[feature_columns])
        target_output_dir = output_dir / target_id
        target_output_dir.mkdir(exist_ok=True)
        for feature in feature_columns:
            feature_rows.append({"target_id": target_id, "feature_column": feature})
        metric_rows, prediction_files = [], []
        for sheet in sheets:
            frame = sheet["frame"]
            sheet_saturation = {str(column) for column in frame.columns if saturation_like_column(column)}
            X_pred_all, _, _ = make_feature_matrix(frame, sheet_saturation)
            for feature in feature_columns:
                if feature not in X_pred_all:
                    X_pred_all[feature] = np.nan
            X_pred = clean_numeric_frame(X_pred_all[feature_columns])
            row_mask = X_pred.notna().any(axis=1)
            if not row_mask.any():
                continue
            X_pred = X_pred.loc[row_mask].reset_index(drop=True)
            source_rows = frame.loc[row_mask].reset_index(drop=True)
            y_pred = model.predict(X_pred)
            prediction = pd.DataFrame({"target_id": target_id, "target_column": target_column, "source_workbook": sheet["workbook"], "source_sheet": sheet["sheet"], "row_index": source_rows.get("row_index", pd.Series(range(len(source_rows))), "y_pred": y_pred, "prediction_status": "scored_unlabeled"})
            if "well_alias" in source_rows:
                prediction["well_alias"] = source_rows["well_alias"]
            if "depth_m" in source_rows:
                prediction["depth_m"] = source_rows["depth_m"]
            if target_column in source_rows:
                y_true = clean_numeric_series(source_rows[target_column])
                valid = y_true.notna()
                prediction.loc[valid, "y_true"] = y_true.loc[valid]
                prediction.loc[valid, "prediction_status"] = "scored_with_target"
                if valid.sum() >= 2:
                    metric_rows.append({"target_id": target_id, "source_workbook": sheet["workbook"], "source_sheet": sheet["sheet"], "rows_scored": int(valid.sum()), "mae": float(mean_absolute_error(y_true.loc[valid], y_pred[valid])), "rmse": float(np.sqrt(mean_squared_error(y_true.loc[valid], y_pred[valid]))), "r2": float(r2_score(y_true.loc[valid], y_pred[valid]))})
            prediction_path = target_output_dir / f"predictions_{sanitize_label(sheet['workbook'])}_{sanitize_label(sheet['sheet'])}.csv"
            prediction.to_csv(prediction_path, index=False)
            prediction_files.append(str(prediction_path))
        pd.DataFrame(metric_rows).to_csv(target_output_dir / "metrics_by_sheet.csv", index=False)
        summary_rows.append({"target_id": target_id, "target_column": target_column, "target_workbook": target["workbook"], "target_sheet": target["sheet_name"], "status": "trained", "alignment_method": alignment_method, "feature_source": feature_source, "training_rows": int(len(X_train)), "feature_count": int(len(feature_columns)), "train_mae": float(mean_absolute_error(y_train, train_pred)), "train_rmse": float(np.sqrt(mean_squared_error(y_train, train_pred))), "train_r2": float(r2_score(y_train, train_pred)) if len(y_train) >= 2 else None, "prediction_file_count": len(prediction_files)})
    run_summary = pd.DataFrame(summary_rows)
    run_summary.to_csv(output_dir / "run_summary.csv", index=False)
    pd.DataFrame(feature_rows).to_csv(output_dir / "feature_columns_by_target.csv", index=False)
    if audit_frames:
        pd.concat(audit_frames, ignore_index=True).to_csv(output_dir / "excluded_feature_columns_by_target.csv", index=False)
    manifest = {"status": "complete", "output_dir": str(output_dir), "target_count": int(len(target_inventory)), "trained_target_count": int((run_summary["status"] == "trained").sum()) if not run_summary.empty else 0, "guardrail": "all saturation-like columns are treated as target-only Y variables"}
    (output_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


## 1. Header scan — run first


In [ ]:
header_result = scan_three_dataset_headers(run_label="notebook_header_scan")
print(json.dumps(header_result, indent=2, default=str))
target_hints_path = Path(header_result["run_dir"]) / "target_header_hints.csv"
target_hints = pd.read_csv(target_hints_path) if target_hints_path.exists() else pd.DataFrame()
display(target_hints.head(50))


## 2. Main three-dataset ML run

Change `TARGET` to an exact target header from the header scan, or leave `auto`.


In [ ]:
TARGET = "auto"
TASK = "auto"  # auto, regression, classification

main_result = run_three_dataset_pipeline(
    requested_target=TARGET,
    requested_task=TASK,
    run_label="notebook_main_three_dataset_run",
)
print(json.dumps(main_result, indent=2, default=str))


## 3. Optional dataset 3 as training

Use this when the target label exists only in `curated_dataset3.xlsx`.


In [ ]:
TARGET_DATASET3 = "auto"

dataset3_result = run_three_dataset_pipeline(
    train_file="curated_dataset3.xlsx",
    test_files=("curated_dataset1.xlsx", "curated_dataset2.xlsx"),
    requested_target=TARGET_DATASET3,
    requested_task="auto",
    run_label="notebook_dataset3_as_training_run",
)
print(json.dumps(dataset3_result, indent=2, default=str))


## 4. Optional all-saturation-like target workflow


In [ ]:
multi_result = fit_and_predict_all_saturations(output_dir=OUTPUT_ROOT / "notebook_all_saturation_targets")
print(json.dumps(multi_result, indent=2, default=str))
summary_path = Path(multi_result.get("output_dir", "")) / "run_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path).head(50))


## 5. Integrated ML output review and export

This replaces the separate output notebook. It reads the trained pipeline outputs and creates paper/slide-ready ML tables and figures under `outputs_runtime/paper_slide_model_exports/`.


In [ ]:
# -----------------------------
# Output review and ML figure export
# -----------------------------

PAPER_EXPORT_ROOT = OUTPUT_ROOT / "paper_slide_model_exports"
PAPER_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

def safe_read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception as exc:
        return {"status": "unreadable", "error": str(exc), "path": str(path)}

# Run inventory from manifests
manifest_rows = []
for path in sorted(OUTPUT_ROOT.rglob("run_manifest.json")):
    payload = safe_read_json(path)
    outputs = payload.get("outputs", {}) if isinstance(payload.get("outputs", {}), dict) else {}
    target = payload.get("target", {}) if isinstance(payload.get("target", {}), dict) else {}
    manifest_rows.append({
        "run_folder": str(path.parent),
        "manifest_path": str(path),
        "status": payload.get("status", ""),
        "blocked_reason": payload.get("blocked_reason", ""),
        "target_column": target.get("column", payload.get("target_column", "")),
        "task": target.get("task", payload.get("task", "")),
        "model_kind": payload.get("model_kind", ""),
        "feature_count": payload.get("feature_count", ""),
        "train_file": payload.get("train_file", ""),
        "test_files": ", ".join(map(str, payload.get("test_files", []))) if isinstance(payload.get("test_files", []), list) else payload.get("test_files", ""),
        "model_path": outputs.get("model", payload.get("model_path", "")),
    })
run_inventory = pd.DataFrame(manifest_rows)
run_inventory.to_csv(PAPER_EXPORT_ROOT / "model_run_inventory.csv", index=False)

# Metrics
metric_frames = []
for run_dir in sorted([p for p in OUTPUT_ROOT.iterdir() if p.is_dir()]):
    for metric_name in ["train_metrics.csv", "test_metrics.csv", "run_summary.csv"]:
        path = run_dir / metric_name
        if path.exists():
            try:
                df = pd.read_csv(path)
                df.insert(0, "run_folder", str(run_dir))
                df.insert(1, "metric_file", metric_name)
                metric_frames.append(df)
            except Exception as exc:
                metric_frames.append(pd.DataFrame([{"run_folder": str(run_dir), "metric_file": metric_name, "read_error": str(exc)}]))
combined_metrics = pd.concat(metric_frames, ignore_index=True, sort=False) if metric_frames else pd.DataFrame()
combined_metrics.to_csv(PAPER_EXPORT_ROOT / "combined_model_metrics.csv", index=False)

# Prediction file inventory only; row-level predictions stay local.
prediction_rows = []
for path in sorted(OUTPUT_ROOT.rglob("predictions_*.csv")):
    try:
        preview = pd.read_csv(path, nrows=5)
        with open(path, "r", encoding="utf-8", errors="ignore") as handle:
            rows = max(sum(1 for _ in handle) - 1, 0)
        prediction_rows.append({
            "prediction_file": str(path),
            "run_folder": str(path.parent),
            "rows": rows,
            "columns": ", ".join(map(str, preview.columns)),
            "target_column_sample": preview["target_column"].dropna().iloc[0] if "target_column" in preview and preview["target_column"].dropna().size else "",
            "dataset_file_sample": preview["dataset_file"].dropna().iloc[0] if "dataset_file" in preview and preview["dataset_file"].dropna().size else "",
            "has_y_true": "y_true" in preview.columns,
            "has_depth": "depth_m" in preview.columns,
            "has_well_alias": "well_alias" in preview.columns,
        })
    except Exception as exc:
        prediction_rows.append({"prediction_file": str(path), "read_error": str(exc)})
prediction_inventory = pd.DataFrame(prediction_rows)
prediction_inventory.to_csv(PAPER_EXPORT_ROOT / "prediction_file_inventory.csv", index=False)

# Feature tables/audits
feature_frames, audit_frames = [], []
for path in sorted(OUTPUT_ROOT.rglob("feature_columns.csv")):
    try:
        df = pd.read_csv(path)
        df.insert(0, "run_folder", str(path.parent))
        feature_frames.append(df)
    except Exception as exc:
        feature_frames.append(pd.DataFrame([{"run_folder": str(path.parent), "read_error": str(exc)}]))
for name in ["feature_policy_audit.csv", "excluded_feature_columns_by_target.csv"]:
    for path in sorted(OUTPUT_ROOT.rglob(name)):
        try:
            df = pd.read_csv(path)
            df.insert(0, "run_folder", str(path.parent))
            df.insert(1, "audit_file", name)
            audit_frames.append(df)
        except Exception as exc:
            audit_frames.append(pd.DataFrame([{"run_folder": str(path.parent), "audit_file": name, "read_error": str(exc)}]))
features_combined = pd.concat(feature_frames, ignore_index=True, sort=False) if feature_frames else pd.DataFrame()
audits_combined = pd.concat(audit_frames, ignore_index=True, sort=False) if audit_frames else pd.DataFrame()
features_combined.to_csv(PAPER_EXPORT_ROOT / "combined_feature_columns.csv", index=False)
audits_combined.to_csv(PAPER_EXPORT_ROOT / "combined_feature_policy_audits.csv", index=False)

# Feature importance from trained random-forest models
importance_rows = []
for model_path in sorted(MODEL_ROOT.rglob("model.joblib")) if MODEL_ROOT.exists() else []:
    try:
        payload = load(model_path)
        model = payload.get("model") if isinstance(payload, dict) else payload
        feature_columns = payload.get("feature_columns", payload.get("features", [])) if isinstance(payload, dict) else []
        target = payload.get("target", {}) if isinstance(payload, dict) else {}
        final_model = model.named_steps.get("model") if hasattr(model, "named_steps") and "model" in model.named_steps else model
        if hasattr(final_model, "feature_importances_") and feature_columns:
            for feature, importance in zip(feature_columns, final_model.feature_importances_):
                importance_rows.append({"model_path": str(model_path), "run_folder": str(model_path.parent), "target_column": target.get("column", target) if isinstance(target, dict) else target, "task": target.get("task", "") if isinstance(target, dict) else "", "feature_column": feature, "importance": float(importance), "importance_type": "random_forest_impurity_importance"})
        else:
            importance_rows.append({"model_path": str(model_path), "run_folder": str(model_path.parent), "feature_column": "", "importance": np.nan, "importance_type": "not_available_for_this_model_type"})
    except Exception as exc:
        importance_rows.append({"model_path": str(model_path), "read_error": str(exc)})
feature_importance = pd.DataFrame(importance_rows)
feature_importance.to_csv(PAPER_EXPORT_ROOT / "model_feature_importance.csv", index=False)

print("Export tables written to:", PAPER_EXPORT_ROOT)
display(run_inventory.head(50))
display(combined_metrics.head(100))
display(prediction_inventory.head(100))
display(feature_importance.head(100))


## 6. End-of-pipeline ML figures

These figures are made from model outputs: metrics, predictions, residuals, depth-profile predictions, and fitted-model feature importance.


In [ ]:
# -----------------------------
# Generate ML output figures
# -----------------------------
figure_manifest = []

def save_current_figure(path, source, figure_type):
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    figure_manifest.append({"figure": str(path), "source": source, "figure_type": figure_type})

# 1. Metrics bar plots
if not combined_metrics.empty:
    for metric in [c for c in ["mae", "rmse", "r2", "accuracy", "balanced_accuracy", "f1_macro"] if c in combined_metrics.columns]:
        plot_df = combined_metrics.dropna(subset=[metric]).copy()
        if plot_df.empty:
            continue
        label_source = plot_df["dataset"].astype(str) if "dataset" in plot_df else plot_df["run_folder"].astype(str).str[-30:]
        split_source = plot_df["split"].astype(str) if "split" in plot_df else plot_df["metric_file"].astype(str)
        plot_df["plot_label"] = label_source + " / " + split_source
        ax = plot_df.plot(kind="bar", x="plot_label", y=metric, legend=False, figsize=(11, 5))
        ax.set_title(f"Model {metric} by dataset/split")
        ax.set_xlabel("Dataset / split")
        ax.set_ylabel(metric)
        plt.xticks(rotation=45, ha="right")
        save_current_figure(PAPER_EXPORT_ROOT / f"figure_model_{metric}.png", "combined_model_metrics.csv", f"metric_bar_{metric}")

# 2. Prediction vs reference and residual plots from prediction files with y_true
prediction_with_truth = []
for row in prediction_inventory.to_dict("records") if not prediction_inventory.empty else []:
    path = Path(row.get("prediction_file", ""))
    if path.exists():
        try:
            df = pd.read_csv(path)
            if {"y_true", "y_pred"}.issubset(df.columns):
                df["prediction_file"] = str(path)
                prediction_with_truth.append(df)
        except Exception:
            pass
pred_truth = pd.concat(prediction_with_truth, ignore_index=True, sort=False) if prediction_with_truth else pd.DataFrame()
if not pred_truth.empty:
    pred_truth["y_true_num"] = pd.to_numeric(pred_truth["y_true"], errors="coerce")
    pred_truth["y_pred_num"] = pd.to_numeric(pred_truth["y_pred"], errors="coerce")
    plot_df = pred_truth.dropna(subset=["y_true_num", "y_pred_num"]).copy()
    if not plot_df.empty:
        ax = plot_df.plot(kind="scatter", x="y_true_num", y="y_pred_num", figsize=(6, 6))
        lo = min(plot_df["y_true_num"].min(), plot_df["y_pred_num"].min())
        hi = max(plot_df["y_true_num"].max(), plot_df["y_pred_num"].max())
        plt.plot([lo, hi], [lo, hi])
        ax.set_title("Predicted vs reference target")
        ax.set_xlabel("Reference / y_true")
        ax.set_ylabel("Predicted / y_pred")
        save_current_figure(PAPER_EXPORT_ROOT / "figure_predicted_vs_reference.png", "predictions_*.csv", "predicted_vs_reference")
        plot_df["residual"] = plot_df["y_pred_num"] - plot_df["y_true_num"]
        ax = plot_df["residual"].plot(kind="hist", bins=30, figsize=(8, 5))
        ax.set_title("Prediction residual distribution")
        ax.set_xlabel("Residual = y_pred - y_true")
        save_current_figure(PAPER_EXPORT_ROOT / "figure_residual_histogram.png", "predictions_*.csv", "residual_histogram")

# 3. Predicted output by depth/well from predictions
prediction_profile_frames = []
for row in prediction_inventory.to_dict("records") if not prediction_inventory.empty else []:
    path = Path(row.get("prediction_file", ""))
    if path.exists():
        try:
            df = pd.read_csv(path)
            if {"y_pred", "depth_m"}.issubset(df.columns):
                df["prediction_file"] = str(path)
                prediction_profile_frames.append(df)
        except Exception:
            pass
profile_df = pd.concat(prediction_profile_frames, ignore_index=True, sort=False) if prediction_profile_frames else pd.DataFrame()
if not profile_df.empty:
    profile_df["depth_num"] = pd.to_numeric(profile_df["depth_m"], errors="coerce")
    profile_df["y_pred_num"] = pd.to_numeric(profile_df["y_pred"], errors="coerce")
    profile_df = profile_df.dropna(subset=["depth_num", "y_pred_num"]).copy()
    if not profile_df.empty:
        # Limit to a readable sample if the output is very large.
        for well, well_df in list(profile_df.groupby(profile_df.get("well_alias", pd.Series(["all"] * len(profile_df))).astype(str)))[:8]:
            well_df = well_df.sort_values("depth_num").head(2000)
            plt.plot(well_df["y_pred_num"], well_df["depth_num"], label=str(well)[:25])
        plt.gca().invert_yaxis()
        plt.title("Predicted model output by depth/well")
        plt.xlabel("Predicted target")
        plt.ylabel("Depth")
        plt.legend(fontsize=8)
        save_current_figure(PAPER_EXPORT_ROOT / "figure_predicted_output_by_depth.png", "predictions_*.csv", "prediction_depth_profile")

# 4. Feature importance
if not feature_importance.empty and "importance" in feature_importance.columns:
    importance = feature_importance.dropna(subset=["importance"]).sort_values("importance", ascending=False).head(20)
    if not importance.empty:
        ax = importance.plot(kind="bar", x="feature_column", y="importance", legend=False, figsize=(11, 5))
        ax.set_title("Top model feature importances")
        ax.set_xlabel("Feature")
        ax.set_ylabel("Importance")
        plt.xticks(rotation=45, ha="right")
        save_current_figure(PAPER_EXPORT_ROOT / "figure_top_feature_importance.png", "model_feature_importance.csv", "feature_importance")

# 5. All-saturation summary
sat_summaries = []
for path in sorted(OUTPUT_ROOT.rglob("run_summary.csv")):
    try:
        df = pd.read_csv(path)
        if "target_id" in df.columns and "status" in df.columns:
            df["summary_file"] = str(path)
            sat_summaries.append(df)
    except Exception:
        pass
sat_summary = pd.concat(sat_summaries, ignore_index=True, sort=False) if sat_summaries else pd.DataFrame()
if not sat_summary.empty:
    sat_summary.to_csv(PAPER_EXPORT_ROOT / "combined_saturation_target_run_summary.csv", index=False)
    trained = sat_summary[sat_summary["status"].astype(str).eq("trained")].copy()
    for metric in [c for c in ["train_mae", "train_rmse", "train_r2"] if c in trained.columns]:
        plot_df = trained.dropna(subset=[metric]).head(30)
        if plot_df.empty:
            continue
        ax = plot_df.plot(kind="bar", x="target_id", y=metric, legend=False, figsize=(12, 5))
        ax.set_title(f"All-saturation target {metric}")
        ax.set_xlabel("Target run")
        ax.set_ylabel(metric)
        plt.xticks(rotation=45, ha="right")
        save_current_figure(PAPER_EXPORT_ROOT / f"figure_all_saturation_{metric}.png", "combined_saturation_target_run_summary.csv", f"all_saturation_{metric}")

figure_manifest_df = pd.DataFrame(figure_manifest)
figure_manifest_df.to_csv(PAPER_EXPORT_ROOT / "figure_manifest.csv", index=False)

deliverables = []
for path in sorted(PAPER_EXPORT_ROOT.glob("*")):
    if path.is_file():
        deliverables.append({"file": str(path), "type": path.suffix.lower().lstrip("."), "bytes": path.stat().st_size})
deliverable_manifest = pd.DataFrame(deliverables)
deliverable_manifest.to_csv(PAPER_EXPORT_ROOT / "paper_slide_deliverable_manifest.csv", index=False)

print("Figure and deliverable exports written to:", PAPER_EXPORT_ROOT)
display(figure_manifest_df)
display(deliverable_manifest)


## What to bring back for paper/slide editing

Use the files in:

```text
outputs_runtime/paper_slide_model_exports/
```

Most useful:
- `combined_model_metrics.csv`
- `model_feature_importance.csv`
- `combined_saturation_target_run_summary.csv`
- `figure_predicted_vs_reference.png`
- `figure_residual_histogram.png`
- `figure_predicted_output_by_depth.png`
- `figure_top_feature_importance.png`
- `figure_model_mae.png`, `figure_model_rmse.png`, `figure_model_r2.png`

Do not upload raw workbook rows, full prediction rows, or fitted `.joblib` model files unless they have been reviewed and reduced to public-safe summaries.
